Task 09 | Streaming Log Inspector

In [ ]:
from datetime import datetime


def read_errors(path, stats=None):
    if stats is not None:
        stats["malformed_lines"] = 0

    with open(path, "r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            parts = line.rstrip("\r\n").split(" | ", 2)

            if len(parts) != 3:
                if stats is not None:
                    stats["malformed_lines"] += 1
                continue

            timestamp, level, message = parts

            try:
                parsed_time = datetime.strptime(timestamp, "%Y-%m-%d %H:%M:%S")
            except ValueError:
                if stats is not None:
                    stats["malformed_lines"] += 1
                continue

            if parsed_time.strftime("%Y-%m-%d %H:%M:%S") != timestamp or not level:
                if stats is not None:
                    stats["malformed_lines"] += 1
                continue

            if level == "ERROR":
                yield line_number, timestamp, message


def summarize_errors(path):
    stats = {}
    counts_by_date = {}

    for line_number, timestamp, message in read_errors(path, stats):
        date = timestamp[:10]
        counts_by_date[date] = counts_by_date.get(date, 0) + 1

    return {
        "errors_by_date": counts_by_date,
        "malformed_lines": stats["malformed_lines"],
    }